# Carnot Heat Pump Screening
This notebook uses the packaged `chocolate_factory.json` case to compare direct versus indirect heat-pump placement and `HPRcycle.CascadeCarnot` versus `HPRcycle.ParallelCarnot` screening. The optimisation is the expensive step, so the notebook creates four named cases, solves each once, and reuses the solved targets for tables and plots.

**Support notice:** This advanced notebook exercises unsupported internal owner modules. Only `from OpenPinch.main import pinch_analysis_service` is compatibility protected.


## Case context
`chocolate_factory.json` is a realistic multi-zone plant case with enough structure to support HPR screening. `PinchWorkspace` keeps the study bundle organized, while each named case still behaves like a real `PinchProblem`. The Carnot backends use base duty scales plus bounded split fractions internally, so trial optimiser points are clipped to process availability before cycle work is evaluated.

In [ ]:
import pandas as pd

from IPython.display import display

from OpenPinch.application.workspace import PinchWorkspace
from OpenPinch.domain.enums import HPRcycle
from OpenPinch.domain._value.resolution import get_scalar_value

PROJECT_NAME = "Chocolate factory"
LOAD_SHARE = 1.0
MULTISTART_COUNT = 10
CONDENSER_COUNT = 3
EVAPORATOR_COUNT = 3

WORKFLOWS = {
    "direct_heat_pump": {
        "label": "Direct Heat Pump",
        "target_name": f"{PROJECT_NAME}/Direct Heat Pump",
        "requires_base_target": False,
    },
    "indirect_heat_pump": {
        "label": "Indirect Heat Pump",
        "target_name": f"{PROJECT_NAME}/Indirect Heat Pump",
        "requires_base_target": True,
    },
}

CYCLE_OPTIONS = {
    "CascadeCarnot": HPRcycle.CascadeCarnot,
    "ParallelCarnot": HPRcycle.ParallelCarnot,
}

In [ ]:
workspace = PinchWorkspace(source="chocolate_factory.json", project_name=PROJECT_NAME)

case_matrix = []
for cycle_key, cycle in CYCLE_OPTIONS.items():
    for workflow_name, workflow in WORKFLOWS.items():
        case_name = f"{cycle_key}__{workflow_name}"
        case = workspace.copy_case(
            source_name="baseline",
            new_name=case_name,
            activate=False,
        )
        case.update_options(
            {
                "HPR_TYPE": cycle.value,
                "HPR_LOAD_MODE": "fraction",
                "HPR_LOAD_FRACTION": LOAD_SHARE,
                "HPR_MAX_MULTISTART": MULTISTART_COUNT,
                "HPR_N_COND": CONDENSER_COUNT,
                "HPR_N_EVAP": EVAPORATOR_COUNT,
            }
        )
        case_matrix.append(
            {
                "case": case_name,
                "cycle": cycle_key,
                "cycle option": cycle.value,
                "workflow": workflow_name,
                "placement": workflow["label"],
                "target name": workflow["target_name"],
            }
        )

pd.DataFrame(case_matrix)

## Solve the four comparison cases once
Each row below is one optimiser run. Keep this cell as the single place where the notebook calls `problem.target.direct_heat_pump()` or `problem.target.indirect_heat_pump()`; later cells reuse `SOLVED_TARGETS` and the cached `comparison` frame.

In [ ]:
SOLVED_TARGETS = {}
rows = []

for spec in case_matrix:
    problem = workspace.case(spec["case"])
    workflow = WORKFLOWS[spec["workflow"]]
    try:
        if workflow["requires_base_target"]:
            problem.target()
        target = getattr(problem.target, spec["workflow"])()
        SOLVED_TARGETS[spec["case"]] = target
        rows.append(
            {
                "cycle": spec["cycle"],
                "placement": spec["placement"],
                "case": spec["case"],
                "target": target.name,
                "Qh": get_scalar_value(getattr(target, "Qh", None)),
                "Qc": get_scalar_value(getattr(target, "Qc", None)),
                "Qr": get_scalar_value(getattr(target, "Qr", None)),
                "hpr_utility_total": get_scalar_value(
                    getattr(target, "hpr_utility_total", None)
                ),
                "hpr_external_utility": get_scalar_value(
                    getattr(target, "hpr_external_utility", None)
                ),
                "hpr_work": get_scalar_value(getattr(target, "hpr_work", None)),
                "hpr_cop": get_scalar_value(getattr(target, "hpr_cop", None)),
                "hpr_success": getattr(target, "hpr_success", None),
                "graph_count": len(problem.plot.catalog()),
                "error": None,
            }
        )
    except Exception as exc:
        rows.append(
            {
                "cycle": spec["cycle"],
                "placement": spec["placement"],
                "case": spec["case"],
                "target": None,
                "Qh": None,
                "Qc": None,
                "Qr": None,
                "hpr_utility_total": None,
                "hpr_external_utility": None,
                "hpr_work": None,
                "hpr_cop": None,
                "hpr_success": False,
                "graph_count": len(problem.plot.catalog()),
                "error": f"{type(exc).__name__}: {exc}",
            }
        )

comparison = pd.DataFrame(rows)
comparison

## Direct versus indirect, Cascade versus Parallel
The tables below reuse `comparison`; they do not trigger additional optimisation calls.

In [ ]:
display(
    comparison.sort_values(["placement", "cycle"])[
        [
            "placement",
            "cycle",
            "Qh",
            "Qc",
            "Qr",
            "hpr_work",
            "hpr_cop",
            "hpr_success",
            "error",
        ]
    ]
)

for metric in ["Qh", "Qc", "Qr", "hpr_work", "hpr_cop"]:
    print(metric)
    display(comparison.pivot(index="placement", columns="cycle", values=metric))

## Inspect one solved HPR-aware plot set
Pick one of the four solved cases and use the internal parent-owned plot accessors. This cell reads the solved target and graph catalog from the previous optimisation cell.

In [ ]:
SELECTED_CASE = "CascadeCarnot__direct_heat_pump"
selected_problem = workspace.case(SELECTED_CASE)
selected_target = SOLVED_TARGETS[SELECTED_CASE]
selected_zone_name = comparison.loc[comparison["case"] == SELECTED_CASE, "placement"].iloc[0]

print(
    {
        "case": SELECTED_CASE,
        "target": selected_target.name,
        "hpr_cop": get_scalar_value(selected_target.hpr_cop),
    }
)
display(selected_problem.plot.net_load_profiles_with_heat_pump(zone_name=selected_zone_name))
display(selected_problem.plot.grand_composite_curve_with_heat_pump(zone_name=selected_zone_name))

## Which graph families appear after each solved workflow?
Use `plot.catalog()` together with the comparison table to see which graph families become available after each target is solved. This stays on the parent-owned graph catalog instead of depending on lower-level cycle internals.

In [ ]:
graph_families = []
for spec in case_matrix:
    problem = workspace.case(spec["case"])
    comparison_row = comparison.loc[comparison["case"] == spec["case"]].iloc[0]
    if comparison_row["error"] is not None:
        graph_families.append(
            {
                "placement": spec["placement"],
                "cycle": spec["cycle"],
                "case": spec["case"],
                "graph families": None,
                "graph count": comparison_row["graph_count"],
                "status": comparison_row["error"],
            }
        )
        continue
    catalog = problem.plot.catalog()
    graph_families.append(
        {
            "placement": spec["placement"],
            "cycle": spec["cycle"],
            "case": spec["case"],
            "graph families": ", ".join(sorted(catalog["Graph Type"].unique())),
            "graph count": len(catalog),
            "status": "ok",
        }
    )

pd.DataFrame(graph_families)

## Optional case-to-case deltas
`compare_cases()` is useful when the two cases contain the same target name. The example below compares the two direct heat-pump cycle choices without running another optimisation.

In [ ]:
workspace.compare_cases(
    "CascadeCarnot__direct_heat_pump",
    "ParallelCarnot__direct_heat_pump",
    target_name=WORKFLOWS["direct_heat_pump"]["target_name"],
    base_label="CascadeCarnot direct",
    other_label="ParallelCarnot direct",
)

## Interpretation
A useful HPR option improves the utility picture while keeping the cycle work and COP operationally plausible. In this workflow, `PinchWorkspace` keeps the four named study cases, the comparison frame shows placement and cycle effects side by side, and the standard plot accessors show how each solved HPR target changes the net-load and GCC views.